# Farm X: layered milk forecast, herd × yield per cow (PFOI experiment)

The same structure as PFOI's engine (herd first, then milk per cow), with machine learning in
each layer:

| Layer | Forecasts (1–90 days ahead) | Learns from |
|---|---|---|
| **1. Herd** | milking cows | every day 2015–2026 (herd counts have no placeholder or shipped milk) |
| **2. Yield** | clean litres per recorded cow | clean milk days only, plus layer 1's forecast for the same day |
| **Combined** | herd milk = layer 1 × layer 2 | — |

Each layer picks LightGBM or Random Forest on a validation part of its own training data
(never the test). Every layer and the combination are scored on the same held-out 20 % against
persistence (carrying the last value forward). The change of every forecast splits exactly into
a herd part and a yield part (§6).

**No leakage between layers.** Layer 2 learns from layer 1 forecasts made without the days
they are scored on: for training rows in year *Y*, layer 1 is fitted only on herd rows that
ended before 1 January *Y*; for test rows, on the herd training rows.

**Experiment status:** `pfoi-nb-layers-0.1-dev`, notebook only; nothing is written to the
database. Privacy as notebook 02: herd totals per day only; placeholder and shipped milk are
never used for modelling (AGENTS.md, docs/model_input2.md §5). Exports go to
`logs/pfoi/notebook_ml/` (gitignored). **Clear all outputs before committing.**

In [ ]:
USE_SYNTHETIC = False        # True = fictional demo data, no database connection

# ---- fixed before any run (do not tune these on test results) ------------------------
INPUT_SCHEMA      = "model_input2"
TARGET            = "per_cow"     # layer 2 target (the loading cell of notebook 02 builds it)
CLEAN_MAX_LITRES  = 18.0          # a day above 18 L per recorded cow is removed
MIN_RECORDED_COWS = 20            # a yield day counts only if ≥ 20 cows have a clean record
MODEL_FROM        = None          # e.g. "2026-04-01": yield forecast dates only from here
HORIZON_DAYS      = 90            # 1, 2 and 3 months ahead
TEST_SHARE        = 0.20          # last 20 % of yield forecast dates = test (herd uses the same date)
HERD_ORIGIN_STEP  = 3             # herd forecast dates every 3 days (long history)
YIELD_ORIGIN_STEP = 1             # yield forecast dates every day (few clean days)
VALID_SHARE       = 0.25          # last 25 % of each layer's training dates pick its model
TOLERANCE         = 0.10
TARGET_HIT        = 0.85
MONTH_MIN_COVERAGE = 0.80         # a 30-day block is scored if ≥ 80 % of its days are recorded
CHART_DATES       = None          # forecast dates for the test charts; None = automatic
SEED              = 17
MODEL_VERSION     = "pfoi-nb-layers-0.1-dev"
LGBM_CHOICE       = "lightgbm_a"  # LightGBM setting (as notebook 02)

import sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

COMPONENT_DIR = Path.cwd().resolve()
if COMPONENT_DIR.name == "notebooks":
    COMPONENT_DIR = COMPONENT_DIR.parent
sys.path.insert(0, str(COMPONENT_DIR))
OUT_DIR = COMPONENT_DIR / "logs" / "pfoi" / "notebook_ml"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE = "demo" if USE_SYNTHETIC else "model"

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams.update({"figure.figsize": (13, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = {"actual": "#3b3b3b", "persistence": "#a0a0a0", "herd_only": "#2a6fdb",
       "yield_only": "#3aa66a", "layered": "#b8572a"}
if USE_SYNTHETIC:
    print("SYNTHETIC DEMO DATA: every number below is fictional (source = 'demo').")

## 1. Import dataset and cleaning

Identical to notebook 02 §1 / §1b (same query, same cleaning, same rules).

In [ ]:
PANEL_COLS = ["n_herd", "milking", "dry_open", "dry_pregnant", "heifers_open", "heifers_pregnant",
              "calves", "calvings", "dry_offs", "real_abortions", "deaths", "culls", "sales", "entries",
              "is_sparse_day", "identified_share",
              "observed_milking_cows", "lactating_unrecorded", "milk_observed_records",
              "thi_mean", "rain_total_mm", "weather_is_complete",
              "mastitis_episodes", "n_sessions", "n_blood_alarms", "n_conductivity_alarms",
              # milk columns of model_input2 (herd totals per day)
              "milk_observed_litres", "milk_observed_cow_litres", "milk_observed_placeholder_litres",
              "milk_excluded_high_records", "milk_excluded_high_litres", "milk_reassigned_to_unidentified",
              "milk_record_type", "herd_milk_measured", "herd_milk_is_estimated", "milk_per_recorded_cow",
              "is_implausible_volume"]

PANEL_SQL = f'''
SELECT date, {", ".join(PANEL_COLS)}
FROM {{schema}}.day_panel ORDER BY date
'''

# milking-cow composition per day: shares only (herd level)
CELL_SQL = '''
SELECT date,
       SUM(n_animals)::float                                                   AS cells_milking,
       SUM(n_animals) FILTER (WHERE parity_group = '1')::float                AS par_1,
       SUM(n_animals) FILTER (WHERE parity_group = '2')::float                AS par_2,
       SUM(n_animals) FILTER (WHERE parity_group = '3+')::float               AS par_3p,
       SUM(n_animals) FILTER (WHERE dim_band = '0-60')::float                 AS dim_0_60,
       SUM(n_animals) FILTER (WHERE dim_band = '61-150')::float               AS dim_61_150,
       SUM(n_animals) FILTER (WHERE dim_band = '151-305')::float              AS dim_151_305,
       SUM(n_animals) FILTER (WHERE dim_band = '305+')::float                 AS dim_305p
FROM {schema}.day_cell WHERE state = 'milking'
GROUP BY date ORDER BY date
'''

def synthetic_milk_columns(p: pd.DataFrame, dates: pd.DatetimeIndex, milking: np.ndarray,
                            per_cow: np.ndarray, r) -> None:
    # fictional model_input2 milk columns: cow samples 2020-22 and from 2026-04, herd totals 2025-09 .. 2026-06
    n = len(dates)
    rec_share = np.select([(dates >= "2020-01-01") & (dates < "2022-03-01"),
                           (dates >= "2022-03-01") & (dates < "2025-09-01"),
                           (dates >= "2025-09-01") & (dates < "2026-04-01"),
                           dates >= "2026-04-01"], [0.2, 0.01, 0.001, 0.6], 0.0)
    cows = np.round(milking * rec_share * r.uniform(0.7, 1.3, n))
    high = r.binomial(cows.astype(int), 0.03).astype(float)
    cow_l = cows * per_cow
    cow_l[r.random(n) < 0.01] *= 2.2                                       # days with doubled records
    ph = (dates >= "2025-09-01") & (dates < "2026-07-01")
    ph_l = np.where(ph, milking * per_cow * 0.95, np.nan)
    p["observed_milking_cows"] = cows
    p["milk_observed_cow_litres"] = np.where(cows > 0, cow_l, np.nan)
    p["milk_observed_placeholder_litres"] = ph_l
    p["milk_excluded_high_records"] = high
    p["milk_excluded_high_litres"] = high * 24.0
    p["milk_observed_litres"] = np.nansum([p["milk_observed_cow_litres"], ph_l, high * 24.0], axis=0)
    p["milk_observed_records"] = cows + high + ph
    herd_total = ph & (p["milk_observed_litres"] >= 5 * milking)
    sample = ~herd_total & (cows >= 10)
    p["milk_record_type"] = np.where(herd_total, "herd_total", np.where(sample, "cow_sample", "none"))
    p["herd_milk_measured"] = np.where(herd_total, p["milk_observed_litres"], np.nan)
    p["herd_milk_is_estimated"] = sample
    p["milk_per_recorded_cow"] = np.where(cows >= 10, cow_l / np.maximum(cows, 1), np.nan)
    p["milk_reassigned_to_unidentified"] = np.where(herd_total, high * 24.0, np.nan)
    p["identified_share"] = np.where(p["milk_observed_litres"] > 0,
                                     np.nan_to_num(p["milk_observed_cow_litres"]) / p["milk_observed_litres"], np.nan)
    p["target_litres"] = np.where(herd_total, p["herd_milk_measured"],
                                  np.where(sample, p["milk_per_recorded_cow"] * milking, np.nan))
    p["is_implausible_volume"] = p["target_litres"] / milking > 18
    p["is_scoreable"] = ~np.isnan(p["target_litres"]) & ~p["is_implausible_volume"]
    p["data_gap"] = np.isnan(p["target_litres"])


def synthetic_inputs(seed: int = 7) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Fictional herd shaped like the real contract: DelPro milk from 2020-08, a thin
    # 2023-24 record period, rich sensor data in 2025-26. For testing the notebook only.
    r = np.random.default_rng(seed)
    dates = pd.date_range("2015-08-01", "2026-09-30", freq="D")
    n = len(dates)
    t = np.arange(n)
    doy = dates.dayofyear.values
    milking = np.clip(170 + np.cumsum(r.normal(0, 0.8, n)) * 0.6 + 20 * np.sin(2 * np.pi * t / 1500), 90, 260)
    per_cow = 8.5 - 0.8 * np.sin(2 * np.pi * (doy - 60) / 365) + r.normal(0, 0.35, n)
    per_cow[(dates >= "2021-06-01") & (dates < "2022-10-01")] -= 1.2       # a yield dip
    litres = milking * per_cow * np.where(dates.weekday == 6, 0.97, 1.0)
    p = pd.DataFrame({"date": dates.date})
    p["milking"] = milking.round()
    p["n_herd"] = (milking * 2.1 + r.normal(0, 5, n)).round()
    p["dry_open"] = (milking * 0.10).round(); p["dry_pregnant"] = (milking * 0.18 + 6 * np.sin(t / 60)).round()
    p["heifers_open"] = (milking * 0.25).round(); p["heifers_pregnant"] = (milking * 0.15).round()
    p["calves"] = (milking * 0.35).round()
    for c, lam in [("calvings", 0.5), ("dry_offs", 0.45), ("real_abortions", 0.03), ("deaths", 0.05),
                   ("culls", 0.08), ("sales", 0.04), ("entries", 0.4)]:
        p[c] = r.poisson(lam, n)
    has = dates >= "2020-08-01"
    thin = (dates >= "2023-01-01") & (dates < "2025-01-01")
    has = has & ~(thin & (r.random(n) < 0.55))
    has = has & ~((dates >= "2022-03-01") & (dates < "2022-04-15"))      # an outage
    p["target_litres"] = np.where(has, litres, np.nan)
    p["is_scoreable"] = has
    p["data_gap"] = ~has
    p["is_sparse_day"] = False
    p["identified_share"] = np.where(has, np.clip(0.85 + r.normal(0, 0.05, n), 0, 1), np.nan)
    p["observed_milking_cows"] = np.where(has, milking * np.where(thin, 0.5, 0.95), 0).round()
    p["lactating_unrecorded"] = (milking * np.where(thin, 0.4, 0.05)).round()
    p["milk_observed_records"] = p["observed_milking_cows"]
    thi = 74 + 3 * np.sin(2 * np.pi * (doy - 100) / 365) + r.normal(0, 1, n)
    p["thi_mean"] = thi; p["rain_total_mm"] = r.gamma(0.6, 8, n); p["weather_is_complete"] = True
    p["mastitis_episodes"] = np.where(dates < "2019-01-01", r.poisson(0.3, n), r.poisson(0.02, n))
    rich = dates >= "2025-01-01"
    p["n_sessions"] = np.where(rich, milking * 2, np.where(has, milking * 0.3, 0)).round()
    p["n_blood_alarms"] = np.where(rich, r.poisson(30, n), 0)
    p["n_conductivity_alarms"] = np.where(rich, r.poisson(20, n), 0)
    c = pd.DataFrame({"date": dates.date, "cells_milking": milking})
    share1 = 0.35 + 0.05 * np.sin(t / 400)
    c["par_1"] = milking * share1; c["par_2"] = milking * 0.3; c["par_3p"] = milking * (0.35 - 0.05 * np.sin(t / 400))
    fresh = 0.2 + 0.05 * np.sin(2 * np.pi * (doy - 30) / 365)
    c["dim_0_60"] = milking * fresh; c["dim_61_150"] = milking * 0.3
    c["dim_151_305"] = milking * (0.4 - fresh + 0.2); c["dim_305p"] = milking * 0.1
    synthetic_milk_columns(p, dates, milking, per_cow, r)
    return p, c


if USE_SYNTHETIC:
    panel_raw, cells_raw = synthetic_inputs()
else:
    from sqlalchemy import create_engine, text
    from pfoi.config import DatabaseSettings
    settings = DatabaseSettings.from_env()
    engine = create_engine(settings.url, pool_pre_ping=True,
                           connect_args={"options": "-c default_transaction_read_only=on"})
    with engine.connect() as conn:
        panel_raw = pd.read_sql(text(PANEL_SQL.format(schema=INPUT_SCHEMA)), conn)
        cells_raw = pd.read_sql(text(CELL_SQL.format(schema=INPUT_SCHEMA)), conn)
    engine.dispose()

panel = panel_raw.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.set_index("date").asfreq("D")            # one row per calendar day
cells = cells_raw.copy()
cells["date"] = pd.to_datetime(cells["date"])
cells = cells.set_index("date").reindex(panel.index)
for c in ["par_1", "par_2", "par_3p", "dim_0_60", "dim_61_150", "dim_151_305", "dim_305p"]:
    panel["share_" + c] = cells[c] / cells["cells_milking"].replace(0, np.nan)

# never used here: stored target (contains placeholder milk) and shipments
panel = panel.drop(columns=["target_litres", "is_scoreable", "data_gap", "milk_shipped"], errors="ignore")
for c in ("is_sparse_day", "is_implausible_volume", "herd_milk_is_estimated"):
    panel[c] = panel[c].fillna(False).astype(bool)

# §1b cleaning on the dataframe (daily herd totals)
ok = ~panel["is_sparse_day"]
panel["cows_clean"] = panel["observed_milking_cows"].where(ok, 0).fillna(0)
panel["litres_clean"] = panel["milk_observed_cow_litres"].where(ok)
panel["placeholder_litres"] = panel["milk_observed_placeholder_litres"]
panel["records_removed_gt18"] = panel["milk_excluded_high_records"].fillna(0)
panel["litres_removed_gt18"] = panel["milk_excluded_high_litres"].fillna(0)
panel["clean_per_cow"] = panel["litres_clean"] / panel["cows_clean"].replace(0, np.nan)
panel["enough_cows"] = panel["cows_clean"] >= MIN_RECORDED_COWS
panel["per_cow_over_cap"] = panel["enough_cows"] & (panel["clean_per_cow"] > CLEAN_MAX_LITRES)
panel["clean_day"] = panel["enough_cows"] & ~panel["per_cow_over_cap"]
# the series the model learns; every other day is missing (never 0)
if TARGET == "per_cow":
    panel["y"] = panel["clean_per_cow"].where(panel["clean_day"])
    panel["per_cow"] = panel["y"]
elif TARGET == "herd_estimate":
    panel["per_cow"] = panel["clean_per_cow"].where(panel["clean_day"])
    panel["y"] = panel["per_cow"] * panel["milking"].replace(0, np.nan)
else:
    raise ValueError('TARGET must be "per_cow" or "herd_estimate"')

print(f"day_panel: {len(panel):,} days, {panel.index.min():%Y-%m-%d} .. {panel.index.max():%Y-%m-%d}; "
      f"{panel['y'].notna().sum():,} days with a usable target ({TARGET}).")
print(f"cleaning: {panel['records_removed_gt18'].sum():,.0f} cow records > 18 L dropped by model_input2; "
      f"{panel['enough_cows'].sum():,} days with ≥ {MIN_RECORDED_COWS} recorded cows, of which "
      f"{panel['per_cow_over_cap'].sum():,} removed here (> {CLEAN_MAX_LITRES:g} L per recorded cow) "
      f"-> {panel['clean_day'].sum():,} clean days.")

## 2. Features known on the forecast date

The origin features of notebook 02 §3 (every feature uses days before the forecast date).
Layer 1 uses only the herd ones (stocks, pregnancy pipeline, events, lactation mix, weather,
health, season); layer 2 uses all of them, including the recent yield level, plus layer 1's
forecast of the change in milking cows for the target day.

In [ ]:
def roll_mean(s, w):
    return s.rolling(w, min_periods=max(2, w // 4)).mean()


def roll_sum(s, w):
    return s.rolling(w, min_periods=1).sum()


def safe_log_ratio(a, b):
    return np.log(a / b).replace([np.inf, -np.inf], np.nan)


def origin_features(p: pd.DataFrame) -> pd.DataFrame:
    # One row per origin date o; every column uses days < o only.
    past = p.shift(1)                     # value on o-1 sits on row o
    y = past["y"]
    F = pd.DataFrame(index=p.index)
    for w in (7, 28, 91, 365):
        F[f"lvl_{w}"] = roll_mean(y, w)
    F["level_7d"] = F["lvl_7"].fillna(F["lvl_28"])            # the anchor
    F["mom_7_28"] = safe_log_ratio(F["lvl_7"], F["lvl_28"])
    F["mom_28_91"] = safe_log_ratio(F["lvl_28"], F["lvl_91"])
    F["mom_91_365"] = safe_log_ratio(F["lvl_91"], F["lvl_365"])
    F["cv_28"] = y.rolling(28, min_periods=7).std() / F["lvl_28"]
    if TARGET != "per_cow":               # with the per-cow target these equal the level features
        pc = past["per_cow"]
        F["per_cow_7"] = roll_mean(pc, 7)
        F["per_cow_28"] = roll_mean(pc, 28)
        F["per_cow_chg_91"] = safe_log_ratio(F["per_cow_28"], F["per_cow_28"].shift(91))
    for c in ["milking", "n_herd", "dry_pregnant", "dry_open", "heifers_pregnant", "calves"]:
        F[c] = past[c]
        F[f"{c}_chg_28"] = safe_log_ratio(past[c] + 1, past[c].shift(28) + 1)
        F[f"{c}_chg_91"] = safe_log_ratio(past[c] + 1, past[c].shift(91) + 1)
    m = past["milking"].replace(0, np.nan)
    F["pipe_dry_preg_per_milking"] = past["dry_pregnant"] / m
    F["pipe_heifer_preg_per_milking"] = past["heifers_pregnant"] / m
    exits = past["deaths"] + past["culls"] + past["sales"]
    for name, s in [("calvings", past["calvings"]), ("dry_offs", past["dry_offs"]),
                    ("exits", exits), ("entries", past["entries"]), ("abortions", past["real_abortions"])]:
        F[f"{name}_30"] = roll_sum(s, 30)
        F[f"{name}_91"] = roll_sum(s, 91)
    for c in ["share_par_1", "share_par_2", "share_par_3p",
              "share_dim_0_60", "share_dim_61_150", "share_dim_151_305", "share_dim_305p"]:
        F[c] = past[c]
    F["thi_7"] = roll_mean(past["thi_mean"], 7)
    F["thi_28"] = roll_mean(past["thi_mean"], 28)
    F["rain_28"] = roll_sum(past["rain_total_mm"].fillna(0), 28)
    F["mastitis_91"] = roll_sum(past["mastitis_episodes"].fillna(0), 91)
    sess = roll_sum(past["n_sessions"].fillna(0), 28).replace(0, np.nan)
    F["blood_alarm_rate_28"] = roll_sum(past["n_blood_alarms"].fillna(0), 28) / sess
    F["cond_alarm_rate_28"] = roll_sum(past["n_conductivity_alarms"].fillna(0), 28) / sess
    F["sessions_per_cow_28"] = sess / (roll_mean(past["milking"], 28) * 28)
    F["scoreable_share_28"] = y.notna().astype(float).rolling(28, min_periods=1).mean()
    ok = y.notna()
    last_ok = pd.Series(np.where(ok, np.arange(len(ok)), np.nan), index=p.index).ffill()
    F["days_since_gap"] = (pd.Series(np.arange(len(ok)), index=p.index)
                           - pd.Series(np.where(~ok, np.arange(len(ok)), np.nan), index=p.index).ffill())
    F["days_since_record"] = np.arange(len(ok)) - last_ok
    F["recorded_cows_share_28"] = roll_mean(past["cows_clean"] / m, 28)
    F["clean_cows_7"] = roll_mean(past["cows_clean"], 7)
    F["removed_share_28"] = (roll_sum(past["records_removed_gt18"], 28)
                             / roll_sum(past["milk_observed_records"].fillna(0), 28).replace(0, np.nan))
    return F


FEAT = origin_features(panel)
# centred 7-day mean of the series, used for "last year's change over the same window"
Y_C7 = panel["y"].rolling(7, center=True, min_periods=3).mean()
print(f"{FEAT.shape[1]} origin features built for {len(FEAT):,} days.")

In [ ]:
LEADS = np.arange(1, HORIZON_DAYS + 1)
MILK_PREFIXES = ("lvl_", "mom_", "cv_28", "per_cow", "scoreable_share", "days_since_gap",
                 "days_since_record", "recorded_cows_share", "clean_cows", "removed_share")


def refresh_series():
    # series and anchors from the current panel / FEAT (called again if the calendar is extended)
    global herd, yld, herd_anchor, yld_anchor, C7
    herd = panel["milking"].astype(float).where(panel["milking"] > 0)
    yld = panel["y"]                                      # clean litres per recorded cow
    herd_anchor = herd.shift(1)                           # milking cows the day before
    yld_anchor = FEAT["level_7d"]                         # mean of the last 7 clean yield days
    C7 = {"herd": herd.rolling(7, center=True, min_periods=3).mean(), "yield": Y_C7}


refresh_series()
HERD_ORIGIN = [c for c in FEAT.columns if not c.startswith(MILK_PREFIXES) and c != "level_7d"]
YIELD_ORIGIN = [c for c in FEAT.columns if c != "level_7d"]
CAL = ["lead", "tgt_doy_sin", "tgt_doy_cos", "last_year_change"]
L1_FEATURES = HERD_ORIGIN + CAL
L2_FEATURES = YIELD_ORIGIN + CAL + ["tgt_weekday", "l1_change"]


def layer_rows(kind: str, origins: pd.DatetimeIndex, with_target: bool = True) -> pd.DataFrame:
    # origin x lead rows for one layer; target day = origin + lead - 1
    series, anchor, cols = ((herd, herd_anchor, HERD_ORIGIN) if kind == "herd"
                            else (yld, yld_anchor, YIELD_ORIGIN))
    idx = panel.index
    pos = idx.get_indexer(origins)
    o_pos = np.repeat(pos, len(LEADS))
    lead = np.tile(LEADS, len(pos))
    t_pos = o_pos + lead - 1
    t_dates = idx[o_pos] + pd.to_timedelta(lead - 1, unit="D")
    rows = pd.DataFrame(FEAT[cols].to_numpy(dtype=float)[o_pos], columns=cols)
    rows["origin"] = idx[o_pos]
    rows["target_date"] = t_dates
    rows["lead"] = lead
    rows["month_ahead"] = (lead - 1) // 30 + 1
    doy = t_dates.dayofyear.values
    rows["tgt_doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    rows["tgt_doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    rows["tgt_weekday"] = t_dates.weekday.values
    ly_t = C7[kind].reindex(t_dates - pd.Timedelta(days=365)).values
    ly_o = anchor.reindex(idx[o_pos] - pd.Timedelta(days=365)).values
    with np.errstate(divide="ignore", invalid="ignore"):
        rows["last_year_change"] = np.log(ly_t / ly_o)
    rows.loc[~np.isfinite(rows["last_year_change"]), "last_year_change"] = np.nan
    rows["anchor"] = anchor.values[o_pos]
    if with_target:
        inside = t_pos < len(idx)
        actual = np.full(len(rows), np.nan)
        actual[inside] = series.values[t_pos[inside]]
        rows["actual"] = actual
        with np.errstate(divide="ignore", invalid="ignore"):
            rows["y_ratio"] = np.log(rows["actual"] / rows["anchor"])
        h_t = np.full(len(rows), np.nan)
        h_t[inside] = herd.values[t_pos[inside]]
        rows["herd_at_target"] = h_t
    return rows


idx = panel.index
herd_start = herd.first_valid_index() + pd.Timedelta(days=365)
herd_origins = idx[(idx >= herd_start) & herd_anchor.notna().values][::HERD_ORIGIN_STEP]
H = layer_rows("herd", herd_origins)
H = H[np.isfinite(H["y_ratio"])].reset_index(drop=True)

y_start = yld.first_valid_index() + pd.Timedelta(days=60)
if MODEL_FROM is not None:
    y_start = max(y_start, pd.Timestamp(MODEL_FROM))
y_origins = idx[(idx >= y_start) & yld_anchor.notna().values & herd_anchor.notna().values][::YIELD_ORIGIN_STEP]
Y = layer_rows("yield", y_origins)
HX = layer_rows("herd", y_origins)          # layer 1 inputs for the same rows (same order)
keep = np.isfinite(Y["y_ratio"]) & np.isfinite(HX["y_ratio"])
Y, HX = Y[keep].reset_index(drop=True), HX[keep].reset_index(drop=True)

print(f"layer 1 (herd):  {H['origin'].nunique():,} forecast dates, {len(H):,} rows, {len(L1_FEATURES)} features")
print(f"layer 2 (yield): {Y['origin'].nunique():,} forecast dates, {len(Y):,} rows, {len(L2_FEATURES)} features")

## 3. Train / test split (80:20, chronological, one date for both layers)

The split date is the first date of the last 20 % of **yield** forecast dates, and the herd
layer uses the same date, so both layers are tested on the same period. Training rows whose
target day is on or after the split date are removed (purge).

In [ ]:
y_dates = np.sort(Y["origin"].unique())
split_date = pd.Timestamp(y_dates[int(len(y_dates) * (1 - TEST_SHARE))])
H_tr = H[(H["origin"] < split_date) & (H["target_date"] < split_date)].reset_index(drop=True)
H_te = H[H["origin"] >= split_date].reset_index(drop=True)
y_tr_mask = (Y["origin"] < split_date) & (Y["target_date"] < split_date)
y_te_mask = Y["origin"] >= split_date
print(f"split date: {split_date:%Y-%m-%d}")
print(f"herd : train {len(H_tr):,} rows ({H_tr['origin'].min():%Y-%m-%d} .. {H_tr['target_date'].max():%Y-%m-%d}), "
      f"test {len(H_te):,} rows")
print(f"yield: train {int(y_tr_mask.sum()):,} rows, test {int(y_te_mask.sum()):,} rows "
      f"({Y.loc[y_te_mask, 'origin'].min():%Y-%m-%d} .. {Y.loc[y_te_mask, 'target_date'].max():%Y-%m-%d})")

## 4. Models and training

Both layers predict the log ratio to their anchor (milking cows the day before; the 7-day
yield level), with recency weights (half-life 730 days), as in notebook 02. Each layer tries
LightGBM and Random Forest; the last `VALID_SHARE` of its training dates (purged) picks the one
with the higher share within ±10 %. The chosen model is then refitted on all its training rows.

In [ ]:
def recency_weight(dates: pd.Series, ref: pd.Timestamp, half_life: float = 730.0) -> np.ndarray:
    age = (ref - dates).dt.days.clip(lower=0).values
    return 0.5 ** (age / half_life)


def make_ridge():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), Ridge(alpha=10.0))


def make_rf():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestRegressor(n_estimators=200, max_depth=14, min_samples_leaf=30,
                                               max_features=0.5, n_jobs=-1, random_state=SEED))


LGBM_SETTINGS = {
    "lightgbm_a": dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0),
    "lightgbm_b": dict(n_estimators=800, learning_rate=0.02, num_leaves=31, min_child_samples=80,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0),
}


def make_lgbm(name="lightgbm_a", **extra):
    return lgb.LGBMRegressor(random_state=SEED, verbose=-1, **{**LGBM_SETTINGS[name], **extra})


ML_FACTORIES = {"ridge": make_ridge, "random_forest": make_rf,
                "lightgbm_a": lambda: make_lgbm("lightgbm_a"), "lightgbm_b": lambda: make_lgbm("lightgbm_b")}


def fit(factory, tr: pd.DataFrame):
    model = factory()
    w = recency_weight(tr["target_date"], tr["target_date"].max())
    X = tr[MODEL_FEATURES]
    if hasattr(model, "steps"):
        model.fit(X, tr["y_ratio"], **{f"{model.steps[-1][0]}__sample_weight": w})
    else:
        model.fit(X, tr["y_ratio"], sample_weight=w)
    return model


def benchmark_ratio(df: pd.DataFrame) -> dict[str, np.ndarray]:
    return {
        "persistence_7d": np.zeros(len(df)),
        "mean_28d": np.log(df["lvl_28"] / df["level_7d"]).fillna(0).values,
        "seasonal_naive": df["last_year_change"].fillna(0).clip(-0.7, 0.7).values,
    }


def hit_share(pred_ratio, df) -> float:
    pred = df["level_7d"].values * np.exp(pred_ratio)
    return float(np.mean(np.abs(pred / df["actual"].values - 1) <= TOLERANCE))

In [ ]:
CANDIDATES = {"lightgbm": lambda: make_lgbm(LGBM_CHOICE), "random_forest": make_rf}


def fit_xy(factory, X: pd.DataFrame, y: pd.Series, dates: pd.Series):
    model = factory()
    w = recency_weight(dates, dates.max())
    if hasattr(model, "steps"):
        model.fit(X, y, **{f"{model.steps[-1][0]}__sample_weight": w})
    else:
        model.fit(X, y, sample_weight=w)
    return model


def hit(pred_ratio, true_ratio) -> float:
    return float(np.mean(np.abs(np.exp(np.asarray(pred_ratio) - np.asarray(true_ratio)) - 1) <= TOLERANCE))


def pick(df: pd.DataFrame, feats: list[str], name: str):
    o = np.sort(df["origin"].unique())
    cut = pd.Timestamp(o[int(len(o) * (1 - VALID_SHARE))])
    tr = df[(df["origin"] < cut) & (df["target_date"] < cut)]
    va = df[df["origin"] >= cut]
    scores, preds = {"persistence": hit(0.0, va["y_ratio"])}, {}
    for m, factory in CANDIDATES.items():
        preds[m] = fit_xy(factory, tr[feats], tr["y_ratio"], tr["target_date"]).predict(va[feats])
        scores[m] = hit(preds[m], va["y_ratio"])
    best = max(CANDIDATES, key=scores.get)
    print(f"{name}: validation from {cut:%Y-%m-%d} ({len(tr):,} train / {len(va):,} validation rows); "
          + ", ".join(f"{k} {v:.1%}" for k, v in scores.items()) + f" -> {best}")
    return best, va.assign(pred=preds[best])


t0 = time.time()
# layer 1: pick and fit on herd training rows
L1_BEST, l1_valid = pick(H_tr, L1_FEATURES, "layer 1 (herd)")
L1 = fit_xy(CANDIDATES[L1_BEST], H_tr[L1_FEATURES], H_tr["y_ratio"], H_tr["target_date"])

# layer 1 forecasts for the yield rows, out of sample (see top of the notebook)
Y["l1_change"] = np.nan
for yr_ in sorted(Y.loc[y_tr_mask, "origin"].dt.year.unique()):
    cut = min(pd.Timestamp(year=int(yr_), month=1, day=1), split_date)
    tr = H[(H["origin"] < cut) & (H["target_date"] < cut)]
    sel = y_tr_mask & (Y["origin"].dt.year == yr_)
    if len(tr) < 1000:
        print(f"  {yr_}: too little herd history before {cut:%Y-%m-%d}; layer 1 input left empty")
        continue
    m1 = fit_xy(CANDIDATES[L1_BEST], tr[L1_FEATURES], tr["y_ratio"], tr["target_date"])
    Y.loc[sel, "l1_change"] = m1.predict(HX.loc[sel, L1_FEATURES])
Y.loc[y_te_mask, "l1_change"] = L1.predict(HX.loc[y_te_mask, L1_FEATURES])

# layer 2: pick and fit on yield training rows
Y_tr = Y[y_tr_mask].reset_index(drop=True)
L2_BEST, l2_valid = pick(Y_tr, L2_FEATURES, "layer 2 (yield)")
L2 = fit_xy(CANDIDATES[L2_BEST], Y_tr[L2_FEATURES], Y_tr["y_ratio"], Y_tr["target_date"])
print(f"trained in {time.time() - t0:.0f} s")

# 80 % range of the combined forecast from validation errors (layer 1 out of sample + layer 2)
hx_tr = HX[y_tr_mask].reset_index(drop=True)
va_i = l2_valid.index
comb_resid = ((l2_valid["y_ratio"] - l2_valid["pred"])
              + (hx_tr.loc[va_i, "y_ratio"] - Y_tr.loc[va_i, "l1_change"]))
RANGE_Q = (pd.DataFrame({"m": l2_valid["month_ahead"], "r": comb_resid}).dropna()
           .groupby("m")["r"].quantile([0.10, 0.90]).unstack()
           .reindex(range(1, HORIZON_DAYS // 30 + 2)).ffill().bfill())
print("80 % range of the layered forecast (log ratio), by month ahead:"); display(RANGE_Q.round(3))


def range_q(valid: pd.DataFrame) -> pd.DataFrame:
    r = valid["y_ratio"] - valid["pred"]
    return (r.groupby(valid["month_ahead"]).quantile([0.10, 0.90]).unstack()
            .reindex(range(1, HORIZON_DAYS // 30 + 2)).ffill().bfill())


RANGE_L1, RANGE_L2 = range_q(l1_valid), range_q(l2_valid)     # each layer's own range (charts)

## 5. Testing and evaluation (held-out 20 %)

Share within ±10 % (target ≥ 85 %), accuracy = 1 − WAPE, bias and skill against persistence,
by month ahead and grain (daily; weekly = 7-day means, all 7 days recorded; monthly = 30-day
means with ≥ 80 % of days recorded). Four herd-milk forecasts are compared:

| Name | Herd (milking cows) | Yield (litres per cow) | What it shows |
|---|---|---|---|
| `persistence` | last day | last 7 days | benchmark |
| `herd_only` | layer 1 | last 7 days | what layer 1 adds |
| `yield_only` | last day | layer 2 | what layer 2 adds |
| `layered` | layer 1 | layer 2 | the full model |

Herd milk recorded = clean litres per recorded cow × milking cows on the target day (the
`model_input2` §5 estimate, no placeholder).

In [ ]:
def score_models(df: pd.DataFrame, models: list[str], base: str, by: str = "month_ahead") -> pd.DataFrame:
    out = []
    for key, g in df.groupby(by):
        a = g["actual"].values
        base_mae = np.mean(np.abs(g[base].values - a))
        for m in models:
            err = g[m].values - a
            out.append({by: key, "model": m, "n": len(g),
                        "within_10pct": np.mean(np.abs(err / a) <= TOLERANCE),
                        "accuracy_1_wape": 1 - np.abs(err).sum() / np.abs(a).sum(),
                        "bias_pct": err.sum() / a.sum(),
                        "skill_vs_persistence": 1 - np.mean(np.abs(err)) / base_mae if base_mae > 0 else np.nan})
    return pd.DataFrame(out, columns=[by, "model", "n", "within_10pct", "accuracy_1_wape", "bias_pct",
                                      "skill_vs_persistence"])


def blocks(df: pd.DataFrame, size: int, n_blocks: int, cols: list[str], min_cov: float = 1.0) -> pd.DataFrame:
    d = df[df["lead"] <= size * n_blocks].copy()
    d["block"] = (d["lead"] - 1) // size + 1
    g = d.groupby(["origin", "block"])
    out = g[cols + ["actual"]].mean()
    out["n_days"] = g.size()
    return out[out["n_days"] >= int(np.ceil(size * min_cov))].reset_index()


def grains(df: pd.DataFrame, models: list[str], base: str) -> dict[str, pd.DataFrame]:
    wk = blocks(df, 7, 12, models)
    wk["month_ahead"] = (wk["block"] - 1) // 4 + 1
    mo = blocks(df, 30, HORIZON_DAYS // 30, models, MONTH_MIN_COVERAGE).rename(columns={"block": "month_ahead"})
    return {"daily": score_models(df, models, base), "weekly": score_models(wk, models, base),
            "monthly": score_models(mo, models, base)}


def show(scores: dict[str, pd.DataFrame], models: list[str], title: str) -> None:
    for grain, s in scores.items():
        print(f"\n{title}, {grain}: share within ±{TOLERANCE:.0%} by month ahead, %")
        if s.empty:
            print("  (no scored period in the test window)")
        else:
            display((s.pivot(index="model", columns="month_ahead", values="within_10pct").loc[models] * 100).round(1))


# layer 1 alone: milking cows, all herd test dates
l1_te = H_te.assign(persistence=H_te["anchor"],
                    layer1=H_te["anchor"] * np.exp(L1.predict(H_te[L1_FEATURES])))
S1 = grains(l1_te, ["persistence", "layer1"], "persistence")
show(S1, ["persistence", "layer1"], "LAYER 1 milking cows")

# layer 2 alone: litres per recorded cow
Y_te = Y[y_te_mask].reset_index(drop=True)
HX_te = HX[y_te_mask].reset_index(drop=True)
r2 = L2.predict(Y_te[L2_FEATURES])
l2_te = Y_te.assign(persistence=Y_te["anchor"], layer2=Y_te["anchor"] * np.exp(r2))
S2 = grains(l2_te, ["persistence", "layer2"], "persistence")
show(S2, ["persistence", "layer2"], "LAYER 2 litres per recorded cow")

# combined herd milk
herd_f = HX_te["anchor"] * np.exp(Y_te["l1_change"])
yld_f = Y_te["anchor"] * np.exp(r2)
comb = pd.DataFrame({"origin": Y_te["origin"], "target_date": Y_te["target_date"], "lead": Y_te["lead"],
                     "month_ahead": Y_te["month_ahead"],
                     "actual": Y_te["actual"] * Y_te["herd_at_target"],
                     "persistence": HX_te["anchor"] * Y_te["anchor"],
                     "herd_only": herd_f * Y_te["anchor"],
                     "yield_only": HX_te["anchor"] * yld_f,
                     "layered": herd_f * yld_f})
comb = comb[np.isfinite(comb["actual"])].reset_index(drop=True)
comb["lower_80"] = comb["layered"] * np.exp(RANGE_Q[0.10].reindex(comb["month_ahead"]).values)
comb["upper_80"] = comb["layered"] * np.exp(RANGE_Q[0.90].reindex(comb["month_ahead"]).values)
CMODELS = ["persistence", "herd_only", "yield_only", "layered"]
S3 = grains(comb, CMODELS, "persistence")
show(S3, CMODELS, "COMBINED herd milk")
print("\nCOMBINED, daily: skill against persistence (> 0 = better)")
display(S3["daily"].pivot(index="model", columns="month_ahead", values="skill_vs_persistence").loc[CMODELS].round(3))
print("\nCOMBINED, daily: accuracy = 1 - WAPE and bias")
display(S3["daily"].pivot(index="model", columns="month_ahead", values=["accuracy_1_wape", "bias_pct"]).loc[CMODELS].round(3))
cov = ((comb["actual"] >= comb["lower_80"]) & (comb["actual"] <= comb["upper_80"])).groupby(comb["month_ahead"]).mean()
print("\nShare of recorded days inside the layered 80 % range (nominal 0.80):"); display(cov.to_frame("coverage"))

### 5b. Forecasts on unseen test data vs what was recorded

One figure per test forecast date, in the PFOI page style: weekly means of the recorded history,
the recorded outcome after the forecast date (dots, never seen by the models), each layer's
forecast (dashed) and its 80 % range from validation errors. The models are the ones trained
on the training 80 % only. Set `CHART_DATES` in §0 to choose the dates.

In [ ]:
# charts in the PFOI page style: weekly means, recorded history, recorded outcome after the
# forecast date (not seen by the model), the model's forecast (dashed) and its 80 % range
CHART_DIR = OUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)


def weekly_path(values: pd.Series, origin: pd.Timestamp) -> pd.DataFrame:
    # mean per 7-day block counted from the forecast date (negative blocks = history)
    s = values.dropna()
    k = (s.index - origin).days // 7
    g = s.groupby(k)
    out = pd.DataFrame({"mean": g.mean(), "n": g.count()})
    out.index = origin + pd.to_timedelta(out.index * 7, unit="D")
    return out


def forecast_panel(ax, title, history, outcome, fc, lo, hi, origin, label, color, unit):
    h = weekly_path(history, origin)
    h = h[h["n"] >= 4]
    ax.plot(h.index, h["mean"], color="#3b3b3b", lw=1.2, label="Recorded")
    f = weekly_path(fc, origin)
    if lo is not None:
        band = pd.DataFrame({"lo": weekly_path(lo, origin)["mean"], "hi": weekly_path(hi, origin)["mean"]})
        ax.fill_between(band.index, band["lo"], band["hi"], color="#9fb4c7", alpha=0.3, lw=0, label="80 % range")
    ax.plot(f.index, f["mean"], ls="--", color=color, lw=1.6, label=label)
    a = weekly_path(outcome, origin)
    a = a[a["n"] >= 4]
    ax.plot(a.index, a["mean"], marker="o", mfc="white", color="#3b3b3b", lw=0.8, ms=4,
            label="Recorded outcome (not seen by the model)")
    ax.axvline(origin, color="#4a6fa5", ls=":", lw=1)
    ax.set_title(f"{title} ({unit}, weekly mean)", loc="left", fontsize=10)


def chart_dates(origins, outcome_days: pd.Series, wanted=None, n=3, min_days=56):
    # forecast dates for the charts: the ones asked for, else first / middle / last test date
    # that still has ≥ min_days recorded days in its 90-day outcome window
    o = pd.DatetimeIndex(sorted(pd.unique(origins)))
    if wanted:
        return [d for d in pd.DatetimeIndex(wanted) if d in o]
    ok = [d for d in o if outcome_days.loc[d: d + pd.Timedelta(days=HORIZON_DAYS - 1)].notna().sum() >= min_days]
    if not ok:
        return list(o[[0, len(o) // 2, -1]]) if len(o) else []
    pick = sorted({0, len(ok) // 2, len(ok) - 1})
    return [ok[i] for i in pick][:n]



def layered_path(o: pd.Timestamp) -> pd.DataFrame:
    fh = layer_rows("herd", pd.DatetimeIndex([o]), with_target=False)
    fy = layer_rows("yield", pd.DatetimeIndex([o]), with_target=False)
    c1 = L1.predict(fh[L1_FEATURES])
    fy["l1_change"] = c1
    c2 = L2.predict(fy[L2_FEATURES])
    m = fy["month_ahead"]
    q1, q2, q3 = RANGE_L1.reindex(m), RANGE_L2.reindex(m), RANGE_Q.reindex(m)
    out = pd.DataFrame(index=pd.DatetimeIndex(fy["target_date"]))
    out["cows"] = fh["anchor"].values * np.exp(c1)
    out["cows_lo"] = out["cows"] * np.exp(q1[0.10].values); out["cows_hi"] = out["cows"] * np.exp(q1[0.90].values)
    out["yield"] = fy["anchor"].values * np.exp(c2)
    out["yield_lo"] = out["yield"] * np.exp(q2[0.10].values); out["yield_hi"] = out["yield"] * np.exp(q2[0.90].values)
    out["milk"] = out["cows"] * out["yield"]
    out["milk_lo"] = out["milk"] * np.exp(q3[0.10].values); out["milk_hi"] = out["milk"] * np.exp(q3[0.90].values)
    return out


herd_milk = yld * herd
dates_ = chart_dates(Y_te["origin"], yld, CHART_DATES)
for o in dates_:
    f = layered_path(o)
    end = f.index.max()
    hist = slice(o - pd.Timedelta(days=364), o - pd.Timedelta(days=1))
    fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
    forecast_panel(ax[0], "Herd milk = layer 1 × layer 2", herd_milk.loc[hist], herd_milk.loc[o:end],
                   f["milk"], f["milk_lo"], f["milk_hi"], o, "layered", COL["layered"], "litres per day")
    forecast_panel(ax[1], f"Milking cows (layer 1, {L1_BEST})", herd.loc[hist], herd.loc[o:end],
                   f["cows"], f["cows_lo"], f["cows_hi"], o, "layer 1", COL["herd_only"], "cows")
    forecast_panel(ax[2], f"Litres per recorded cow (layer 2, {L2_BEST})", yld.loc[hist], yld.loc[o:end],
                   f["yield"], f["yield_lo"], f["yield_hi"], o, "layer 2", COL["yield_only"], "litres per cow per day")
    ax[0].legend(fontsize=8, loc="upper left", ncol=2)
    ax[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.suptitle(f"Forecast date {o:%Y-%m-%d}: {MODEL_VERSION} on unseen data (herd level; 80 % ranges)" +
                 ("  [SYNTHETIC DEMO]" if USE_SYNTHETIC else ""), fontsize=11)
    plt.tight_layout()
    fig.savefig(CHART_DIR / f"nb03_test_{o:%Y%m%d}{'_demo' if USE_SYNTHETIC else ''}.png", dpi=130)
    plt.show()
print("chart dates:", ", ".join(f"{d:%Y-%m-%d}" for d in dates_), f"| saved to {CHART_DIR}")

## 6. Explainability

**Herd vs yield split.** Every layered forecast is the anchor × herd change × yield change, so
in log terms the change from persistence is exactly `log(layer 1 / cows yesterday) +
log(layer 2 / 7-day yield)`. The table shows the mean signed and absolute size of each part by
month ahead on the test rows: which layer moves the forecast.

**Inside each layer.** LightGBM TreeSHAP (`pred_contrib=True`) summed into factor groups; the
parts add up exactly to each layer's output (checked). If a layer selected Random Forest, its
LightGBM twin, trained on the same rows, is explained.

In [ ]:
parts = pd.DataFrame({"month_ahead": Y_te["month_ahead"], "herd_part": Y_te["l1_change"], "yield_part": r2})
split = parts.groupby("month_ahead").agg(herd_mean=("herd_part", "mean"), yield_mean=("yield_part", "mean"),
                                         herd_abs=("herd_part", lambda x: x.abs().mean()),
                                         yield_abs=("yield_part", lambda x: x.abs().mean()))
split["herd_share_of_change"] = split["herd_abs"] / (split["herd_abs"] + split["yield_abs"])
print("Change from persistence, log units (0.01 ≈ 1 %):"); display(split.round(4))

GROUPS = {
    "recent yield level": ["lvl_", "mom_", "cv_28"],
    "layer 1 forecast": ["l1_change"],
    "herd stocks": ["milking", "n_herd", "dry_open", "calves"],
    "pregnancy pipeline": ["pipe_", "dry_pregnant", "heifers_pregnant"],
    "events": ["calvings_", "dry_offs_", "exits_", "entries_", "abortions_"],
    "lactation mix": ["share_par", "share_dim"],
    "weather": ["thi_", "rain_"],
    "health / sensors": ["mastitis", "alarm", "sessions_per_cow"],
    "record quality": ["scoreable", "days_since", "recorded_cows", "clean_cows", "removed_share"],
    "season / lead": ["tgt_", "lead", "last_year"],
}


def group_of(f: str) -> str:
    for g, keys in GROUPS.items():
        if any(f.startswith(k) or k in f for k in keys):
            return g
    return "other"


def shap_groups(model, X: pd.DataFrame, months: pd.Series) -> pd.DataFrame:
    contrib = model.predict(X, pred_contrib=True)
    assert np.allclose(contrib.sum(axis=1), model.predict(X), atol=1e-6), "SHAP parts do not add up"
    s = pd.DataFrame(np.abs(contrib[:, :-1]), columns=X.columns)
    g = s.T.groupby(s.columns.map(group_of)).sum().T
    g["month_ahead"] = months.values
    imp = g.groupby("month_ahead").mean().T
    return imp.div(imp.sum(axis=0), axis=1)


def lgbm_of(best, model, X, y, d):
    return model if best == "lightgbm" else fit_xy(CANDIDATES["lightgbm"], X, y, d)


s1 = H_te.sample(min(20000, len(H_te)), random_state=SEED)
s2 = Y_te.sample(min(20000, len(Y_te)), random_state=SEED)
imp1 = shap_groups(lgbm_of(L1_BEST, L1, H_tr[L1_FEATURES], H_tr["y_ratio"], H_tr["target_date"]),
                   s1[L1_FEATURES], s1["month_ahead"])
imp2 = shap_groups(lgbm_of(L2_BEST, L2, Y_tr[L2_FEATURES], Y_tr["y_ratio"], Y_tr["target_date"]),
                   s2[L2_FEATURES], s2["month_ahead"])
print("Additivity check passed for both layers.")
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
imp1.sort_values(imp1.columns[0]).plot.barh(ax=ax[0]); ax[0].set_title("Layer 1 (milking cows): share of mean |SHAP|")
imp2.sort_values(imp2.columns[0]).plot.barh(ax=ax[1]); ax[1].set_title("Layer 2 (litres per cow): share of mean |SHAP|")
plt.tight_layout(); plt.show()

## 7. Prediction from the latest data

Both layers are refitted on every row with a known outcome (layer 2 with its out-of-sample
layer 1 inputs), then forecast the next `HORIZON_DAYS` days from the day after the last clean
yield day: milking cows (layer 1), litres per cow (layer 2) and herd milk (layered), daily and
weekly (Monday weeks), with the 80 % range from §4. Exported in the `c1_output.outlook` row
shape to `logs/pfoi/notebook_ml/`.

In [ ]:
L1_full = fit_xy(CANDIDATES[L1_BEST], H[L1_FEATURES], H["y_ratio"], H["target_date"])
Y_known = Y                                   # layer 1 inputs are out of sample (§4)
L2_full = fit_xy(CANDIDATES[L2_BEST], Y_known[L2_FEATURES], Y_known["y_ratio"], Y_known["target_date"])

last_clean = yld.last_valid_index()
origin = last_clean + pd.Timedelta(days=1)
if origin not in panel.index:                  # extend the calendar so the origin row exists
    panel = panel.reindex(pd.date_range(panel.index.min(), origin, freq="D"))
    FEAT = origin_features(panel)
    Y_C7 = panel["y"].rolling(7, center=True, min_periods=3).mean()
    refresh_series()
fy = layer_rows("yield", pd.DatetimeIndex([origin]), with_target=False)
fh = layer_rows("herd", pd.DatetimeIndex([origin]), with_target=False)
fy["l1_change"] = L1_full.predict(fh[L1_FEATURES])
r_y = L2_full.predict(fy[L2_FEATURES])
fut = pd.DataFrame({"target_date": fy["target_date"], "lead": fy["lead"], "month_ahead": fy["month_ahead"],
                    "milking": fh["anchor"] * np.exp(fy["l1_change"]),
                    "milk_per_cow": fy["anchor"] * np.exp(r_y)})
fut["milk_litres"] = fut["milking"] * fut["milk_per_cow"]
q = RANGE_Q.reindex(fut["month_ahead"])
fut["milk_lower"] = fut["milk_litres"] * np.exp(q[0.10].values)
fut["milk_upper"] = fut["milk_litres"] * np.exp(q[0.90].values)
print(f"forecast date {origin:%Y-%m-%d} (last clean yield day {last_clean:%Y-%m-%d}); "
      f"layer 1 {L1_BEST}, layer 2 {L2_BEST} ({MODEL_VERSION}), source = {SOURCE}")
print(f"cows yesterday {fh['anchor'].iloc[0]:,.0f}; 7-day yield {fy['anchor'].iloc[0]:.2f} L per recorded cow")
fut["week_start"] = fut["target_date"] - pd.to_timedelta(fut["target_date"].dt.weekday, unit="D")
cols = ["milking", "milk_per_cow", "milk_litres", "milk_lower", "milk_upper"]
monthly_fc = fut.groupby("month_ahead")[cols].mean()
weekly_fc = fut.groupby("week_start")[cols].mean().assign(days=fut.groupby("week_start").size())
print("Forecast by month ahead (mean per day):"); display(monthly_fc)
print("Weekly forecast (mean per day; first / last week may be partial):"); display(weekly_fc)

hist = (yld * herd).loc[origin - pd.Timedelta(days=120): origin]
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(hist.index, hist.values, color=COL["actual"], lw=0.9, label="recorded (clean estimate)")
ax.fill_between(fut["target_date"], fut["milk_lower"], fut["milk_upper"], color=COL["layered"], alpha=0.18, lw=0, label="80 % range")
ax.plot(fut["target_date"], fut["milk_litres"], color=COL["layered"], lw=1.8, label="layered daily")
ax.step(weekly_fc.index, weekly_fc["milk_litres"], where="post", color="#333", lw=1, ls=":", label="weekly mean")
ax.axvline(origin, color="#888", lw=0.8)
ax.set_title(f"Herd milk: layer 1 × layer 2 from {origin:%Y-%m-%d}" + ("  [SYNTHETIC DEMO]" if USE_SYNTHETIC else ""))
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

quality = lambda h: "high" if h <= 7 else ("moderate" if h <= 30 else "limited")
out_rows = []
for series, col, lo, hi in (("milking", "milking", None, None), ("milk_per_cow", "milk_per_cow", None, None),
                            ("milk_litres", "milk_litres", "milk_lower", "milk_upper")):
    for grain, frame in (("day", fut.set_index("target_date")), ("week", weekly_fc)):
        for i, (period, r) in enumerate(frame.iterrows(), start=1):
            h = int(r["lead"]) if grain == "day" else i
            out_rows.append({"origin_date": f"{origin:%Y-%m-%d}", "target_period": f"{period:%Y-%m-%d}",
                             "grain": grain, "horizon": h, "series": series,
                             "forecast": round(float(r[col]), 2),
                             "lower": round(float(r[lo]), 2) if lo else None,
                             "upper": round(float(r[hi]), 2) if hi else None,
                             "quality_flag": quality(h if grain == "day" else 7 * h),
                             "model_version": MODEL_VERSION, "core_version": INPUT_SCHEMA})

## 8. Summary against the targets

Printed from the tables above. Frontend readiness uses the same fixed rule as notebook 02:
the layered month-1 daily **and** weekly forecasts need ≥ 85 % within ±10 % and positive skill
against persistence.

In [ ]:
def _ok(s_, m) -> bool:
    r = s_[(s_["model"] == m) & (s_["month_ahead"] == 1)]
    return bool(len(r)) and r["within_10pct"].iloc[0] >= TARGET_HIT and r["skill_vs_persistence"].iloc[0] > 0


READY = _ok(S3["daily"], "layered") and _ok(S3["weekly"], "layered")
lines = [f"Layered model: layer 1 = {L1_BEST}, layer 2 = {L2_BEST}   |   test from {split_date:%Y-%m-%d}   |   source = {SOURCE}", ""]
for label, S, m in (("layer 1 milking cows", S1, "layer1"), ("layer 2 litres/cow ", S2, "layer2"),
                    ("layered herd milk   ", S3, "layered")):
    for grain, s_ in S.items():
        t = s_[s_["model"] == m].set_index("month_ahead")
        p = s_[s_["model"] == "persistence"].set_index("month_ahead")
        if t.empty:
            lines.append(f"⚪ {label} {grain:7s}: no scored period in the test window")
        for mo, r in t.iterrows():
            ok = "✅" if r["within_10pct"] >= TARGET_HIT else "❌"
            lines.append(f"{ok} {label} {grain:7s} month {mo}: within ±10 % = {r['within_10pct']:.1%} "
                         f"(persistence {p.loc[mo, 'within_10pct']:.1%}), skill {r['skill_vs_persistence']:+.2f}, n = {int(r['n'])}")
    lines.append("")
lines.append(("✅ FRONTEND READY" if READY else "❌ NOT FRONTEND READY") +
             f": layered month-1 daily and weekly need ≥ {TARGET_HIT:.0%} within ±10 % and positive skill vs persistence")
print("\n".join(lines))

payload = {"source": SOURCE, "model_version": MODEL_VERSION, "layer1": L1_BEST, "layer2": L2_BEST,
           "frontend_ready": bool(READY), "unit": "per day (weekly rows = mean per day)",
           "basis": "herd milk = layer 1 milking cows x layer 2 clean litres per recorded cow; no placeholder or shipped milk",
           "outlook": out_rows}
stamp = f"{origin:%Y%m%d}" + ("_demo" if USE_SYNTHETIC else "")
(OUT_DIR / f"forecast_layers_{stamp}.json").write_text(json.dumps(payload, indent=1))
pd.concat([S.assign(grain=g, layer=lbl) for lbl, SS in (("layer1", S1), ("layer2", S2), ("layered", S3))
           for g, S in SS.items()]).assign(source=SOURCE).to_csv(OUT_DIR / f"test_metrics_layers_{stamp}.csv", index=False)
print("written:", *(p.name for p in sorted(OUT_DIR.glob(f"*layers_{stamp}*"))), sep="\n  ")

**Reporting rules.** Quote the numbers as printed, including any ❌. The thresholds in §0 stay
fixed. Persistence is a benchmark only and is never blended into the forecast.